In [1]:
import os

os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

In [2]:
import os, math, copy
import multiprocessing as mp
import numpy as np
import pinocchio as pin
import simple
from scipy.interpolate import CubicSpline

from simulation_utils import setPhysicsProperties, removeBVHModelsIfAny
from simulation_args  import SimulationArgs
from sim_utils        import setupSimulatorFromArgs
from pin_utils        import addSystemCollisionPairs, addSelectedCollisionPairs

BASE_DIR = '/home/manolis/venvs/Diplomatiki'
URDF_FILE = os.path.join(BASE_DIR, 'leap_description', 'robots', 'leap_right.urdf')
CUBE_URDF = os.path.join(BASE_DIR, 'cube_description', 'cube.urdf')

model, geom_model, _ = pin.buildModelsFromUrdf(URDF_FILE, [BASE_DIR])
NQ_HAND = model.nq   # 16
NV_HAND = model.nv   # 16

cube_pos0 = np.array([-0.0576, -0.0296,  0.192])
print(f'Cube centre pos0  : {np.round(cube_pos0, 4)}')

cube_model, cube_geom_model, _ = pin.buildModelsFromUrdf(
    CUBE_URDF, [BASE_DIR], pin.JointModelFreeFlyer())

# hand + cube σε ένα μοντέλο, parent=0 → cube free-floating
f_model, f_geom_model = pin.appendModel(
    model, cube_model, geom_model, cube_geom_model, 0, pin.SE3.Identity())

#HERE
print("All geometry objects in f_geom_model:")
for i, g in enumerate(f_geom_model.geometryObjects):
    print(f"  {i:3d}: {g.name}  (parent joint: {g.parentJoint})")

removeBVHModelsIfAny(f_geom_model)
setPhysicsProperties(f_geom_model, material="wood", compliance=0.0)

NQ_TOTAL = f_model.nq   # 23
NV_TOTAL = f_model.nv   # 22

Q_LO_HAND = model.lowerPositionLimit.copy(); Q_HI_HAND = model.upperPositionLimit.copy()
V_MAX_HAND = model.velocityLimit.copy(); TAU_MAX = model.effortLimit.copy()

q0 = pin.neutral(f_model)
v0 = np.zeros(NV_TOTAL)
q0[NQ_HAND : NQ_HAND + 3] = cube_pos0
q0[NQ_HAND +3 :] = np.array([0.0682,  0.0125, -0.0798,  0.9944])


#addSystemCollisionPairs(f_model, f_geom_model, q0)

addSelectedCollisionPairs(f_model, f_geom_model, q0,include_geom_substrings=['fingertip_0', 'thumb_fingertip', 'dip_0', 'cube'])

data_full = f_model.createData()
gdata_full = f_geom_model.createData()



INDEX_TIP_JOINT_ID = f_model.getJointId('joint_3')
THUMB_TIP_JOINT_ID = f_model.getJointId('joint_15')
CUBE_JOINT_ID = f_model.getJointId("root_joint")
print(f'INDEX_TIP_JOINT = {INDEX_TIP_JOINT_ID}  THUMB_TIP_JOINT = {THUMB_TIP_JOINT_ID}')

try:
    import viser
    from viser.extras import ViserUrdf
    from yourdfpy import URDF

    def xyzw_to_wxyz(a): return np.concatenate((a[-1:], a[:3]))

    # αλλαγή σειράς joints: pinocchio q[0:16] → viser config
    def map_pin_vis(q):
        return np.array((q[1], q[0], q[2], q[3],
                         q[9], q[8], q[10], q[11],
                         q[13], q[12], q[14], q[15],
                         q[4], q[5], q[6], q[7]))

    def _vis_fname(fname):
        f = fname
        if f.startswith('package://'): return os.path.join(BASE_DIR, f[len('package://'):])
        if not os.path.isabs(f): return os.path.join(os.path.dirname(URDF_FILE), f)
        return f

    urdf_hand_obj = URDF.load(URDF_FILE,
        build_scene_graph=True, build_collision_scene_graph=True,
        load_meshes=True, load_collision_meshes=True,
        mesh_dir=BASE_DIR, filename_handler=_vis_fname)
    urdf_cube_obj = URDF.load(CUBE_URDF,
        build_scene_graph=True, build_collision_scene_graph=True,
        load_meshes=True, load_collision_meshes=True)

    server = viser.ViserServer()
    server.scene.add_frame('/robot_frame', wxyz=(1,0,0,0), position=(0,0,0), show_axes=False)
    server.scene.add_grid('/grid', position=(0,0,0), width=2., height=2.)

    viser_urdf = ViserUrdf(server, urdf_hand_obj, root_node_name='/robot_frame',
                            load_meshes=True, load_collision_meshes=True)
    viser_urdf.show_visual = True; viser_urdf.show_collision = False

    cube_frame = server.scene.add_frame('/cube_frame',
        wxyz=(0.9303, -0.0555, -0.3583,  0.0559), position=tuple(cube_pos0), show_axes=False)
    viser_cube = ViserUrdf(server, urdf_cube_obj, root_node_name='/cube_frame',
                            load_meshes=True, load_collision_meshes=True)
    viser_cube.show_visual = True; viser_cube.show_collision = False

    def update_viser_from_q(q_full):
        viser_urdf.update_cfg(map_pin_vis(q_full[:NQ_HAND]))
        qc = q_full[NQ_HAND : NQ_HAND + 7]
        cube_frame.position = tuple(qc[:3])
        cube_frame.wxyz = tuple(xyzw_to_wxyz(qc[3:]))

    update_viser_from_q(q0)
    VISER_AVAILABLE = True
    print('Viser started — http://localhost:8080')
except ImportError:
    VISER_AVAILABLE = False
    def update_viser_from_q(q_full): pass
    print('viser not found — visualisation disabled.')


Cube centre pos0  : [-0.0576 -0.0296  0.192 ]
All geometry objects in f_geom_model:
    0: palm_lower_0  (parent joint: 0)
    1: mcp_joint_0  (parent joint: 1)
    2: pip_0  (parent joint: 2)
    3: dip_0  (parent joint: 3)
    4: fingertip_0  (parent joint: 4)
    5: pip_4_0  (parent joint: 5)
    6: thumb_pip_0  (parent joint: 6)
    7: thumb_dip_0  (parent joint: 7)
    8: thumb_fingertip_0  (parent joint: 8)
    9: mcp_joint_2_0  (parent joint: 9)
   10: pip_2_0  (parent joint: 10)
   11: dip_2_0  (parent joint: 11)
   12: fingertip_2_0  (parent joint: 12)
   13: mcp_joint_3_0  (parent joint: 13)
   14: pip_3_0  (parent joint: 14)
   15: dip_3_0  (parent joint: 15)
   16: fingertip_3_0  (parent joint: 16)
   17: cube_link_0  (parent joint: 17)
Whitelisted geometries (5):
    3: dip_0
    4: fingertip_0
    7: thumb_dip_0
    8: thumb_fingertip_0
   17: cube_link_0
Num col pairs =  8
INDEX_TIP_JOINT = 4  THUMB_TIP_JOINT = 8


╭────── viser (listening *:8081) ───────╮
│             ╷                         │
│   HTTP      │ http://localhost:8081   │
│   Websocket │ ws://localhost:8081     │
│             ╵                         │
╰───────────────────────────────────────╯

Viser started — http://localhost:8080


(viser) Connection opened (0, 1 total), 223 persistent messages

In [ ]:
args = SimulationArgs()
args.dt = 0.005; args.Kp = 100; args.Kd = 100
args.tol = 1e-4; args.tol_rel = 1e-3; args.maxit = 500
args.warm_start = 1; args.mu_prox = 1e-3
args.contact_solver = 'ADMM'; args.admm_update_rule = 'spectral'
args.max_contacts_per_pair = 4; args.patch_tolerance = 5e-3
args.safe_distance = 5e-3  # gap κάτω από το οποίο ενεργοποιείται collision
args.debug = False         # stat_record off; turn on only when diagnosing
args.solve_ccp = True     # NCP — physically accurate Coulomb cone if false cpp is faster though
args.epa_tolerance = 1e-3
args.gjk_tolerance = 1e-3



sim = simple.Simulator(f_model, data_full, f_geom_model, gdata_full)
setupSimulatorFromArgs(sim, gdata_full, args)
sim.reset()

DT = args.dt

In [49]:
# index finger (0-3) + thumb (4-7)
INDEX_IDX = np.array([0, 1, 2, 3])
THUMB_IDX = np.array([4, 5, 6, 7])
CONTROL_IDX = np.concatenate([INDEX_IDX, THUMB_IDX])
NON_CONTROL_IDX = np.array([i for i in range(NQ_HAND) if i not in CONTROL_IDX])

Q_PREGRASP = np.zeros(NQ_HAND)
Q_PREGRASP[0] = 0.29;  Q_PREGRASP[1] = 0.02
Q_PREGRASP[2] = 0.93;  Q_PREGRASP[3] = 0.51
Q_PREGRASP[4] = 1.59;  Q_PREGRASP[5] = 0.0
Q_PREGRASP[6] = -0.75; Q_PREGRASP[7] = 1.04

q0[:NQ_HAND] = Q_PREGRASP
v0 = np.zeros(NV_TOTAL)
update_viser_from_q(q0)

# offset μετρημένο οπτικά από URDF mesh
INDEX_CONTACT_OFFSET_LOCAL = np.array([-0.01, -0.05,  0.015])
THUMB_CONTACT_OFFSET_LOCAL = np.array([-0.01, -0.06, -0.013])

def index_contact_pos(model_, data_, q):
    pin.forwardKinematics(model_, data_, q)
    oMi = data_.oMi[INDEX_TIP_JOINT_ID]
    return oMi.translation + oMi.rotation @ INDEX_CONTACT_OFFSET_LOCAL

def thumb_contact_pos(model_, data_, q):
    pin.forwardKinematics(model_, data_, q)
    oMi = data_.oMi[THUMB_TIP_JOINT_ID]
    return oMi.translation + oMi.rotation @ THUMB_CONTACT_OFFSET_LOCAL

Ts, zeta = 0.1, 0.7
_data_pd = model.createData()
M_mat = np.asarray(pin.crba(model, _data_pd, pin.neutral(model)))
M_eff = np.diag(M_mat)
wn = 4.0 / (zeta * Ts)
KP_vec = wn**2 * M_eff
KD_vec = np.maximum(2.0 * zeta * wn * M_eff - model.damping, 0.0)
# low-pass: ALPHA_FILT στο reference, BETA_FILT στην ταχύτητα
ALPHA_FILT = 0.7; BETA_FILT = 0.5

# CUBE_HALF_SIZE πρέπει να ταιριάζει με cube.urdf
CUBE_HALF_SIZE = 0.035; CUBE_MASS = 0.02   # kg

LOCAL_FACE_CANDIDATES = np.array([
    [ CUBE_HALF_SIZE,  0.0,            0.0            ],  # 0: +X
    [-CUBE_HALF_SIZE,  0.0,            0.0            ],  # 1: -X
    [ 0.0,             CUBE_HALF_SIZE, 0.0            ],  # 2: +Y
    [ 0.0,            -CUBE_HALF_SIZE, 0.0            ],  # 3: -Y
    [ 0.0,             0.0,            CUBE_HALF_SIZE ],  # 4: +Z
    [ 0.0,             0.0,           -CUBE_HALF_SIZE ],  # 5: -Z
])

def cube_position_from_q(q):
    return q[NQ_HAND : NQ_HAND + 3].copy()

def cube_rotation_from_q(q):
    # pinocchio freeflyer αποθηκεύει quaternion ως xyzw
    quat_xyzw = q[NQ_HAND + 3 : NQ_HAND + 7]
    quat_obj = pin.Quaternion(quat_xyzw[3], quat_xyzw[0], quat_xyzw[1], quat_xyzw[2])
    return quat_obj.toRotationMatrix()

# antipodal pinch: index +X face, thumb -X face
IDX_FACE = 0; THUMB_FACE = 1
P_INDEX_CUBE_LOCAL = LOCAL_FACE_CANDIDATES[IDX_FACE].copy()
P_THUMB_CUBE_LOCAL = LOCAL_FACE_CANDIDATES[THUMB_FACE].copy()
_N_IDX_LOCAL = P_INDEX_CUBE_LOCAL / np.linalg.norm(P_INDEX_CUBE_LOCAL)
_N_TH_LOCAL  = P_THUMB_CUBE_LOCAL / np.linalg.norm(P_THUMB_CUBE_LOCAL)

print(f'Index face : {IDX_FACE}   local center = {np.round(P_INDEX_CUBE_LOCAL, 4)}')
print(f'Thumb face : {THUMB_FACE}   local center = {np.round(P_THUMB_CUBE_LOCAL, 4)}')
_dot = float(np.dot(_N_IDX_LOCAL, _N_TH_LOCAL))
print(f'Normal dot = {_dot:.3f}  (should be -1.0 for antipodal pinch)')
if _dot > -0.8:
    print('WARNING: faces are not well-opposed — check Q_PREGRASP!')

def index_target_world(q):
    return cube_position_from_q(q) + cube_rotation_from_q(q) @ P_INDEX_CUBE_LOCAL

def thumb_target_world(q):
    return cube_position_from_q(q) + cube_rotation_from_q(q) @ P_THUMB_CUBE_LOCAL

def index_face_normal_world(q):
    return cube_rotation_from_q(q) @ _N_IDX_LOCAL

def thumb_face_normal_world(q):
    return cube_rotation_from_q(q) @ _N_TH_LOCAL

def signed_face_distance(p_tip, p_face_world, n_face_world):
    return float(np.dot(p_tip - p_face_world, n_face_world))



K_SAMPLES = 50; T_HORIZON = 60; N_EXEC_STEPS = 1
ALPHA_MPPI = 1.0; MAX_ITER = 400
N_WORKERS = max(1, os.cpu_count())
chunksize = math.ceil(K_SAMPLES / N_WORKERS)
chunksize = 1

RNG_SEED = 7
rng = np.random.default_rng(RNG_SEED)
N_KNOTS = 4
KNOT_TIMES = np.linspace(0, T_HORIZON - 1, N_KNOTS)
FULL_TIMES = np.arange(T_HORIZON)


W_POINT_TRACK = 100000.0    # finger-on-cube-face tracking
W_PINCH       = 50000.0     # gap between fingers matches gap between targets

# Contact maintenance 
W_NO_CONTACT  = 200000000.0    # quadratic penalty on positive face-gap (asymmetric)
W_NO_FORCE    = 5000.0      # one-shot penalty when force is near zero in contact phase

# Cube stillness
W_CUBE_DRIFT  = 2000.0     # cube position drift from reference (3rd priority)
W_CUBE_VEL    = 10000.0    # cube linear velocity 
W_CUBE_OMEGA  = 2000.0        # cube angular velocity 

# Regularization — low priority, just smoothing
W_POSTURE     = 0.0       # hand pose stays near saved pinch posture
W_VEL         = 0.05        # finger joint velocities (you already had this)
W_DU          = 0.001       # control noise penalty (you already had this)

# Terminal weights — scale up the running weights to enforce end-state quality
W_TERM_POINT_TRACK = 5.0 * W_POINT_TRACK
W_TERM_PINCH       = 5.0 * W_PINCH
W_TERM_NO_CONTACT  = 5.0 * W_NO_CONTACT
W_TERM_CUBE_DRIFT  = 5.0 * W_CUBE_DRIFT
W_TERM_CUBE_VEL    = 5.0 * W_CUBE_VEL
W_TERM_CUBE_OMEGA  = 5.0 * W_CUBE_OMEGA

# gravity staging: [0, -3, -6, -9.81] m/s², streak_required iterations για advance
GRAVITY_STAGES = [0.0, -3.0, -6.0, -9.81]
GRAVITY_STREAK_REQUIRED = 20
ENABLE_GRAVITY_STAGING = True

PINCH_E_TOL = 10e-3; PINCH_G_TOL = 20e-3
CUBE_VC_TOL = 0.01; CUBE_WC_TOL = 0.1

PROX_SCALE = 0.015

print(f'MPPI: K={K_SAMPLES}  T={T_HORIZON}  N_exec={N_EXEC_STEPS}  workers={N_WORKERS}')

print(f'Gravity stages: {GRAVITY_STAGES}  streak_required={GRAVITY_STREAK_REQUIRED}  staging={ENABLE_GRAVITY_STAGING}')
print(f'W_POINT_TRACK={W_POINT_TRACK:.0f}  W_PINCH={W_PINCH:.0f}')

def build_cubic_basis():
    basis = np.zeros((N_KNOTS, T_HORIZON))

    for i in range(N_KNOTS):
        y = np.zeros(N_KNOTS)
        y[i] = 1.0
        cs = CubicSpline(KNOT_TIMES, y, bc_type='natural')
        basis[i, :] = cs(FULL_TIMES)

    return basis

CUBIC_BASIS = build_cubic_basis()



INDEX_IDX = np.array([0, 1, 2, 3])
THUMB_IDX = np.array([4, 5, 6, 7])
q0[INDEX_IDX] = np.array([1.1543, -0.3415,  0.5771, 0.0])
q0[THUMB_IDX] = np.array([ 1.4326,  0.1852, -0.7874,  0.6])

_data_check = f_model.createData()
_p_idx_ik = index_contact_pos(f_model, _data_check, q0)
_p_th_ik  = thumb_contact_pos(f_model, _data_check, q0)
_sd_idx_ik = signed_face_distance(_p_idx_ik, index_target_world(q0), index_face_normal_world(q0))
_sd_th_ik  = signed_face_distance(_p_th_ik,  thumb_target_world(q0),  thumb_face_normal_world(q0))

update_viser_from_q(q0)


Index face : 0   local center = [0.035 0.    0.   ]
Thumb face : 1   local center = [-0.035  0.     0.   ]
Normal dot = -1.000  (should be -1.0 for antipodal pinch)
MPPI: K=50  T=60  N_exec=1  workers=16
Gravity stages: [0.0, -3.0, -6.0, -9.81]  streak_required=20  staging=True
W_POINT_TRACK=100000  W_PINCH=50000


In [ ]:
_w_model = _w_data_kin = _w_sim  = None


def init_worker(fm, fgm, sim_args):
    global _w_model, _w_data_kin, _w_sim
    _w_model = fm.copy()
    _fgm_loc = fgm.copy()
    _w_data_kin = _w_model.createData()
    _w_data_sim = _w_model.createData()
    _w_geom = _fgm_loc.createData()
    _w_sim = simple.Simulator(_w_model, _w_data_sim, _fgm_loc, _w_geom)
    setupSimulatorFromArgs(_w_sim, _w_geom, sim_args)
    _w_sim.reset()
    

'''
def _worker_pd(q, v, q_des_hand, q_ref, v_filt, gravity_z):
    if not np.isfinite(q).all() or not np.isfinite(v).all():
        return np.zeros(NV_HAND), q_ref.copy(), v_filt.copy()
    q_ref_new = ALPHA_FILT * q_ref + (1.0 - ALPHA_FILT) * q_des_hand
    v_filt_new = BETA_FILT * v_filt + (1.0 - BETA_FILT) * v[:NV_HAND]
    q_safe = np.clip(q[:NQ_HAND], Q_LO_HAND, Q_HI_HAND)
    v_safe = np.clip(v_filt_new, -V_MAX_HAND, V_MAX_HAND)
    g_total = pin.computeGeneralizedGravity(_w_model, _w_data_kin, q)
    err = q_ref_new - q_safe
    tau = g_total[:NV_HAND] + KP_vec * err - KD_vec * v_safe
    return np.clip(tau, -TAU_MAX, TAU_MAX), q_ref_new, v_filt_new
'''

def _worker_pd(q, v, q_des_hand, gravity_z):
    if not np.isfinite(q).all() or not np.isfinite(v).all():
        return np.zeros(NV_HAND)
    
    q_safe = np.clip(q[:NQ_HAND], Q_LO_HAND, Q_HI_HAND)
    v_safe = np.clip(v[:NV_HAND], -V_MAX_HAND, V_MAX_HAND)
    g_total = pin.computeGeneralizedGravity(_w_model, _w_data_kin, q)
    err = q_des_hand[:NQ_HAND] - q_safe
    tau = g_total[:NV_HAND] + KP_vec * err - KD_vec * v_safe
    return np.clip(tau, -TAU_MAX, TAU_MAX)



R = cube_rotation_from_q(q0)
cube_pos = cube_position_from_q(q0)
p_i_0  = cube_pos + R @ P_INDEX_CUBE_LOCAL
p_th_0 = cube_pos + R @ P_THUMB_CUBE_LOCAL
print(cube_pos)

#
pin.forwardKinematics(f_model, data_full, q0)
    
oMi_cube = data_full.oMi[CUBE_JOINT_ID]
R         = oMi_cube.rotation
cube_pos  = oMi_cube.translation
print(cube_pos)

def _worker_running_cost(q, v, du):
    
    pin.forwardKinematics(_w_model, _w_data_kin, q)
    
    
    oMi_cube = _w_data_kin.oMi[CUBE_JOINT_ID]
    R         = oMi_cube.rotation
    cube_pos  = oMi_cube.translation

    # Fingertip contact points (offset from frame origin to actual pad center)
    oMi_idx = _w_data_kin.oMi[INDEX_TIP_JOINT_ID]
    oMi_th  = _w_data_kin.oMi[THUMB_TIP_JOINT_ID]
    p_i  = oMi_idx.act(INDEX_CONTACT_OFFSET_LOCAL)
    p_th = oMi_th.act(THUMB_CONTACT_OFFSET_LOCAL)

    # Cube-face target points in world
    p_i_star  = cube_pos + R @ P_INDEX_CUBE_LOCAL
    p_th_star = cube_pos + R @ P_THUMB_CUBE_LOCAL

    # Errors 
    e_i  = p_i  - p_i_star
    e_th = p_th - p_th_star
    e_gap = e_i - e_th     

    n_idx_world = R @ _N_IDX_LOCAL
    n_th_world  = R @ _N_TH_LOCAL
    sd_idx = (p_i  - p_i_star).dot(n_idx_world)
    sd_th  = (p_th - p_th_star).dot(n_th_world)

    # Asymmetric gap penalty: zero when in contact (gap ≤ 5mm), grows with positive gap
    NO_CONTACT_THRESHOLD = 5e-3
    gap_idx = max(0.0, sd_idx - NO_CONTACT_THRESHOLD)
    gap_th  = max(0.0, sd_th  - NO_CONTACT_THRESHOLD)
    cost_gap = W_NO_CONTACT * (gap_idx**2 + gap_th**2)

    cost_track = W_POINT_TRACK * (e_i.dot(e_i) + e_th.dot(e_th))
    cost_pinch = W_PINCH * e_gap.dot(e_gap)

    v_ctrl   = v[CONTROL_IDX]      # CONTROL_IDX is already < NV_HAND, the [:NV_HAND] slice is redundant
    cost_vel = W_VEL * v_ctrl.dot(v_ctrl)
    cost_du  = W_DU  * du[CONTROL_IDX].dot(du[CONTROL_IDX])

    v_c_local     = v[NV_HAND : NV_HAND + 3]
    omega_c_local = v[NV_HAND + 3 : NV_HAND + 6]
    v_c_world     = R @ v_c_local
    omega_c_world = R @ omega_c_local

    cost_cube_vel   = W_CUBE_VEL   * v_c_world.dot(v_c_world)
    cost_cube_omega = W_CUBE_OMEGA * omega_c_world.dot(omega_c_world)

    # 2. Cube position drift from reference
    cube_drift = cube_pos - cube_pos0
    cost_cube_drift = W_CUBE_DRIFT * cube_drift.dot(cube_drift)

    
    return cost_track + cost_pinch + cost_vel + cost_du + cost_gap + cost_cube_vel + cost_cube_omega + cost_cube_drift


def _worker_phi(q, v):
    pin.forwardKinematics(_w_model, _w_data_kin, q)
    
    
    oMi_cube = _w_data_kin.oMi[CUBE_JOINT_ID]
    R         = oMi_cube.rotation
    cube_pos  = oMi_cube.translation

 
    oMi_idx = _w_data_kin.oMi[INDEX_TIP_JOINT_ID]
    oMi_th  = _w_data_kin.oMi[THUMB_TIP_JOINT_ID]
    p_i  = oMi_idx.act(INDEX_CONTACT_OFFSET_LOCAL)
    p_th = oMi_th.act(THUMB_CONTACT_OFFSET_LOCAL)

    p_i_star  = cube_pos + R @ P_INDEX_CUBE_LOCAL
    p_th_star = cube_pos + R @ P_THUMB_CUBE_LOCAL

    e_i  = p_i  - p_i_star
    e_th = p_th - p_th_star
    e_gap = e_i - e_th

    cost_track = W_TERM_POINT_TRACK * (e_i.dot(e_i) + e_th.dot(e_th))
    cost_pinch = W_TERM_PINCH * e_gap.dot(e_gap)

    # Terminal velocity penalty — punish ending the rollout in motion
    v_c_local     = v[NV_HAND     : NV_HAND + 3]
    omega_c_local = v[NV_HAND + 3 : NV_HAND + 6]
    v_c_world     = R @ v_c_local
    omega_c_world = R @ omega_c_local
    cost_term_vel = W_TERM_CUBE_VEL   * v_c_world.dot(v_c_world)
    cost_term_omg = W_TERM_CUBE_OMEGA * omega_c_world.dot(omega_c_world)

    # Terminal drift penalty
    cube_drift = cube_pos - cube_pos0
    cost_term_drift = W_TERM_CUBE_DRIFT * cube_drift.dot(cube_drift)


    return cost_track + cost_pinch + cost_term_drift + cost_term_vel + cost_term_omg

def expand_knot_noise_cubic(eps_knots):
    K = eps_knots.shape[0]
    out = np.zeros((K, NQ_HAND, T_HORIZON))

    eps_ctrl = np.einsum(
        'kjn,nt->kjt',
        eps_knots,
        CUBIC_BASIS
    )

    out[:, CONTROL_IDX, :] = eps_ctrl
    return out


def rollout_worker(task):
    state_1d, U_nom_flat, eps_flat, gravity_z = task
    _w_model.gravity = pin.Motion(np.array([0.0, 0.0, gravity_z, 0.0, 0.0, 0.0]))
    U = U_nom_flat.reshape(NQ_HAND, T_HORIZON)
    E = eps_flat.reshape(NQ_HAND, T_HORIZON)
    q = state_1d[:NQ_TOTAL].copy()
    v = state_1d[NQ_TOTAL:].copy()
    q_ref = q[:NQ_HAND].copy()
    v_filt = v[:NV_HAND].copy()
    _w_sim.reset()
    total = 0.0
    for t in range(T_HORIZON):
        u, du = U[:, t], E[:, t]
        q_cmd_hand = np.clip(u + du, Q_LO_HAND, Q_HI_HAND)
        tau_hand = _worker_pd(q, v, q_cmd_hand, gravity_z)
        tau_total = np.zeros(NV_TOTAL)
        tau_total[:NV_HAND] = tau_hand
        try:
            _w_sim.step(q, v, tau_total, DT)
        except Exception:
            return 1e12
        q, v = _w_sim.qnew.copy(), _w_sim.vnew.copy()
        if (not np.all(np.isfinite(q)) or not np.all(np.isfinite(v))
                or np.max(np.abs(v)) > 100.0):
            return 1e12
        total += _worker_running_cost(q, v, du)
    return total + _worker_phi(q, v)


def mppi_update(U_nom, eps_K, costs, alpha, convergence_scale = 1.0):
    valid = np.isfinite(costs) & (costs < 1e11)
    if valid.sum() < 4:
        return U_nom
    cv = costs[valid]
    ev = eps_K[valid]
    beta = float(cv.min())
    # IQR temperature, robust σε outliers
    lam = max(1e-6, 0.15 * float(np.percentile(cv, 80) - np.percentile(cv, 20)))
    w = np.exp(-(cv - beta) / lam)
    w /= w.sum() + 1e-12
    d_soft = np.einsum('k,kdt->dt', w, ev)
    # elite blend: top 15% με ίσο βάρος
    ne = max(4, int(0.15 * len(cv)))
    d_eli = ev[np.argsort(cv)[:ne]].mean(axis=0)
    # 0.55/0.45 δουλεύει καλύτερα από 0.5/0.5
    return U_nom + alpha * convergence_scale * (0.55 * d_soft + 0.45 * d_eli)


def pd_grav_main(q, v, q_des_hand, q_ref, v_filt, gravity_z):
    if not np.isfinite(q).all() or not np.isfinite(v).all():
        return np.zeros(NV_HAND), q_ref.copy(), v_filt.copy(), True
    q_ref_new = ALPHA_FILT * q_ref + (1.0 - ALPHA_FILT) * q_des_hand
    v_filt_new = BETA_FILT * v_filt + (1.0 - BETA_FILT) * v[:NV_HAND]
    q_safe = np.clip(q[:NQ_HAND], Q_LO_HAND, Q_HI_HAND)
    v_safe = np.clip(v_filt_new, -V_MAX_HAND, V_MAX_HAND)
    g_total = pin.computeGeneralizedGravity(f_model, data_full, q)
    err = q_ref_new - q_safe
    tau = g_total[:NV_HAND] + KP_vec * err - KD_vec * v_safe
    return np.clip(tau, -TAU_MAX, TAU_MAX), q_ref_new, v_filt_new, False


[-0.0576 -0.0296  0.192 ]
[-0.0576 -0.0296  0.192 ]


In [51]:
import time

U_nom = np.tile(q0[:NQ_HAND].reshape(NQ_HAND, 1), (1, T_HORIZON))

gravity_stage_idx = 0
gravity_streak = 0
current_gravity_z = GRAVITY_STAGES[gravity_stage_idx]

f_model.gravity = pin.Motion(np.array([0.0, 0.0, current_gravity_z, 0.0, 0.0, 0.0]))

# 2 s pre-stabilization ώστε να κατασταλούν τα transients επαφής
sim.reset()
print('Stabilizing for 2 s (gravity off, holding pose) ...')
N_PRESTEPS = int(2.0 / DT)
q_stab = q0.copy()
v_stab = v0.copy()
q_ref_stab = q_stab[:NQ_HAND].copy()
v_filt_stab = np.zeros(NV_HAND)

for _pre in range(N_PRESTEPS):
    q_cmd_stab = np.clip(q0[:NQ_HAND], Q_LO_HAND, Q_HI_HAND)
    tau_s, q_ref_stab, v_filt_stab, bad = pd_grav_main(
        q_stab, v_stab, q_cmd_stab, q_ref_stab, v_filt_stab, gravity_z=0.0)
    if bad: break
    tau_pre = np.zeros(NV_TOTAL)
    tau_pre[:NV_HAND] = tau_s
    sim.step(q_stab, v_stab, tau_pre, DT)
    q_stab, v_stab = sim.qnew.copy(), sim.vnew.copy()
    if not np.all(np.isfinite(q_stab)) or not np.all(np.isfinite(v_stab)):
        q_stab, v_stab = q0.copy(), v0.copy()
        break

update_viser_from_q(q_stab)
print(f'Stabilization done ({N_PRESTEPS} steps).')
print(f'  Cube pos : {np.round(cube_position_from_q(q_stab), 4)}')
print(f'  Max |v|  : {np.max(np.abs(v_stab)):.4f} rad/s')

state = np.concatenate([q_stab, v_stab])
q_ref_exec = q_stab[:NQ_HAND].copy()
v_filt_exec = v_stab[:NV_HAND].copy()

_h_idx_pad = _h_th_pad = _h_idx_tgt = _h_th_tgt = None
if VISER_AVAILABLE:
    try:
        _h_idx_pad = server.scene.add_icosphere('/mppi/index_pad',    radius=0.004,
            color=(0.0, 0.8, 0.2), position=tuple(index_contact_pos(f_model, data_full, q_stab)))
        _h_th_pad  = server.scene.add_icosphere('/mppi/thumb_pad',    radius=0.004,
            color=(0.0, 0.4, 1.0), position=tuple(thumb_contact_pos(f_model, data_full, q_stab)))
        _h_idx_tgt = server.scene.add_icosphere('/mppi/index_target', radius=0.004,
            color=(1.0, 0.5, 0.0), position=tuple(index_target_world(q_stab)))
        _h_th_tgt  = server.scene.add_icosphere('/mppi/thumb_target', radius=0.004,
            color=(0.7, 0.0, 0.9), position=tuple(thumb_target_world(q_stab)))
        print('Viser: green=index pad, blue=thumb pad, orange=index target, purple=thumb target')
    except Exception as _ve:
        print(f'Viser markers skipped: {_ve}')

def _update_viser_dots(q):
    if not VISER_AVAILABLE: return
    try:
        if _h_idx_pad is not None: _h_idx_pad.position = tuple(index_contact_pos(f_model, data_full, q))
        if _h_th_pad  is not None: _h_th_pad.position  = tuple(thumb_contact_pos(f_model, data_full, q))
        if _h_idx_tgt is not None: _h_idx_tgt.position = tuple(index_target_world(q))
        if _h_th_tgt  is not None: _h_th_tgt.position  = tuple(thumb_target_world(q))
    except Exception:
        pass

_update_viser_dots(q_stab)

print(f'\nStarting MPPI: K={K_SAMPLES}  T={T_HORIZON}  N_exec={N_EXEC_STEPS}  workers={N_WORKERS}')
print(f'Gravity stage 0/{len(GRAVITY_STAGES)-1}: g={current_gravity_z:.1f} m/s^2')

with mp.Pool(N_WORKERS, initializer=init_worker,
             initargs=(f_model, f_geom_model, args)) as pool:

    for it in range(MAX_ITER):
        t0 = time.perf_counter()

        q_it = state[:NQ_TOTAL]
        v_it = state[NQ_TOTAL:]

        pin.forwardKinematics(f_model, data_full, q_it)
        

        oMi_idx_cur = data_full.oMi[INDEX_TIP_JOINT_ID]
        oMi_th_cur  = data_full.oMi[THUMB_TIP_JOINT_ID]
        r_idx_cur = oMi_idx_cur.rotation @ INDEX_CONTACT_OFFSET_LOCAL
        r_th_cur  = oMi_th_cur.rotation  @ THUMB_CONTACT_OFFSET_LOCAL
        p_idx_cur = oMi_idx_cur.translation + r_idx_cur
        p_th_cur  = oMi_th_cur.translation  + r_th_cur
        p_i_star_cur  = index_target_world(q_it)
        p_th_star_cur = thumb_target_world(q_it)

        e_i_cur  = p_idx_cur - p_i_star_cur
        e_th_cur = p_th_cur  - p_th_star_cur
        ei_norm  = float(np.linalg.norm(e_i_cur))
        eth_norm = float(np.linalg.norm(e_th_cur))
        g_act_cur = p_idx_cur  - p_th_cur
        g_des_cur = p_i_star_cur - p_th_star_cur
        pinch_err = float(np.linalg.norm(g_act_cur - g_des_cur))
        vc_norm   = float(np.linalg.norm(v_it[NV_HAND     : NV_HAND + 3]))
        wc_norm   = float(np.linalg.norm(v_it[NV_HAND + 3 : NV_HAND + 6]))

        R_cur = cube_rotation_from_q(q_it)
        R_ref = cube_rotation_from_q(q0)
        c_rot = (np.trace(R_ref.T @ R_cur) - 1) / 2
        cube_rot_deg = float(np.degrees(np.arccos(np.clip(c_rot, -1.0, 1.0))))

        # antithetic sampling: K/2 + mirror
        K_HALF = K_SAMPLES // 2
        # Δυνα
        if vc_norm < 0.03 and wc_norm < 0.2 and pinch_err < 20e-3:
            noise_std = 0.01
            conv_scale = 0.2      
        elif vc_norm < 0.04 and wc_norm < 0.3 and pinch_err < 20e-3:
            noise_std = 0.03
            conv_scale = 0.4     
        elif vc_norm < 0.06 and wc_norm < 0.4 and pinch_err < 20e-3:
            noise_std = 0.05
            conv_scale = 0.8     
        else:
            noise_std = 0.1    
            conv_scale = 1.0
        eps_half = noise_std * rng.standard_normal((K_HALF, len(CONTROL_IDX), N_KNOTS))
        eps_knots = np.concatenate([eps_half, -eps_half], axis=0)
        if eps_knots.shape[0] < K_SAMPLES:
            eps_knots = np.concatenate([
                eps_knots,
                noise_std * rng.standard_normal(
                    (K_SAMPLES - eps_knots.shape[0], len(CONTROL_IDX), N_KNOTS))
            ], axis=0)
        eps_K = expand_knot_noise_cubic(eps_knots)
        eps_K[0] = 0.0   # sample 0 = nominal rollout

        state_flat = state.copy()
        U_nom_flat = U_nom.ravel().copy()
        eps_K_flat = eps_K.reshape(K_SAMPLES, -1)

        tasks = [
            (state_flat, U_nom_flat, eps_K_flat[k], current_gravity_z)
            for k in range(K_SAMPLES)
        ]

        costs = np.array(
            pool.map(rollout_worker, tasks, chunksize=chunksize),
            dtype=float
        )
        
        valid_mask = np.isfinite(costs) & (costs < 1e11)
        n_valid = int(valid_mask.sum())
        if n_valid >= 4:
            U_nom = mppi_update(U_nom, eps_K, costs, ALPHA_MPPI, conv_scale)
            U_nom = np.clip(U_nom, Q_LO_HAND[:, None], Q_HI_HAND[:, None])
            U_nom[NON_CONTROL_IDX, :] = q0[NON_CONTROL_IDX, None]

        cv = costs[valid_mask]
        best_cost = float(cv.min())  if n_valid > 0 else float('nan')
        mean_cost = float(cv.mean()) if n_valid > 0 else float('nan')
        t_iter = time.perf_counter() - t0

        
        print(
            f'[{it:3d}] '
            f'g={current_gravity_z:.1f}  '
            f'S({gravity_streak}/{GRAVITY_STREAK_REQUIRED})  '
            f'e_i={ei_norm*1e3:5.1f}mm  '
            f'e_th={eth_norm*1e3:5.1f}mm  '
            f'pinch={pinch_err*1e3:5.1f}mm  '
            f'vc={vc_norm:.3f}m/s  '
            f'wc={wc_norm:.3f}rad/s  '
            f'rot={cube_rot_deg:.1f}deg  '
            f'best={best_cost:.1f}  mean={mean_cost:.1f}  '
            f'{1/t_iter:.1f}Hz'
        )
           
        q_ex = state[:NQ_TOTAL].copy()
        v_ex = state[NQ_TOTAL:].copy()
        
        diverged = False
        for step_i in range(N_EXEC_STEPS):
            q_cmd_exec = np.clip(U_nom[:, step_i], Q_LO_HAND, Q_HI_HAND)
            tau_hand, q_ref_exec, v_filt_exec, bad = pd_grav_main(
                q_ex, v_ex, q_cmd_exec, q_ref_exec, v_filt_exec, gravity_z=current_gravity_z)
            if bad:
                diverged = True; break
            tau_total_exec = np.zeros(NV_TOTAL)
            tau_total_exec[:NV_HAND] = tau_hand
            sim.step(q_ex, v_ex, tau_total_exec, DT)
            q_ex, v_ex = sim.qnew.copy(), sim.vnew.copy()
            if not np.all(np.isfinite(q_ex)) or not np.all(np.isfinite(v_ex)):
                diverged = True; break

        if diverged:
            print(f'[{it:3d}] DIVERGED — resetting velocities')
            v_ex[:] = 0.0
            q_ref_exec = q_ex[:NQ_HAND].copy()
            v_filt_exec = v_ex[:NV_HAND].copy()

        state = np.concatenate([q_ex, v_ex])
        update_viser_from_q(q_ex)
        _update_viser_dots(q_ex)

        # receding horizon
        U_nom = np.hstack([U_nom[:, N_EXEC_STEPS:],
                           np.tile(U_nom[:, -1:], (1, N_EXEC_STEPS))])

        q_post = state[:NQ_TOTAL]
        v_post = state[NQ_TOTAL:]

        pin.forwardKinematics(f_model, data_full, q_post)
        oMi_ip = data_full.oMi[INDEX_TIP_JOINT_ID]
        oMi_tp = data_full.oMi[THUMB_TIP_JOINT_ID]
        p_i_p  = oMi_ip.translation + oMi_ip.rotation @ INDEX_CONTACT_OFFSET_LOCAL
        p_th_p = oMi_tp.translation + oMi_tp.rotation @ THUMB_CONTACT_OFFSET_LOCAL
        tgt_i_p  = index_target_world(q_post)
        tgt_th_p = thumb_target_world(q_post)
        ei_p  = float(np.linalg.norm(p_i_p  - tgt_i_p))
        eth_p = float(np.linalg.norm(p_th_p - tgt_th_p))
        g_a_p = p_i_p - p_th_p
        g_d_p = tgt_i_p - tgt_th_p
        pg_p  = float(np.linalg.norm(g_a_p - g_d_p))
        vc_p  = float(np.linalg.norm(v_post[NV_HAND     : NV_HAND + 3]))
        wc_p  = float(np.linalg.norm(v_post[NV_HAND + 3 : NV_HAND + 6]))

        pinch_ok = (vc_p<0.05 and wc_p<0.5 and ei_p<15e-3 and eth_p < 15e-3)

        if pinch_ok: gravity_streak += 1
        else: gravity_streak = 0

        if ENABLE_GRAVITY_STAGING:
            if (gravity_streak >= GRAVITY_STREAK_REQUIRED
                    and gravity_stage_idx < len(GRAVITY_STAGES) - 1):
                noise_std = 0.25
                gravity_stage_idx += 1
                current_gravity_z = GRAVITY_STAGES[gravity_stage_idx]
                gravity_streak = 0
                f_model.gravity = pin.Motion(
                    np.array([0.0, 0.0, current_gravity_z, 0.0, 0.0, 0.0]))
                print(f'>>> Gravity stage -> {gravity_stage_idx}: '
                      f'g={current_gravity_z:.2f} m/s^2')
        else:
            if gravity_streak >= GRAVITY_STREAK_REQUIRED:
                print(f'>>> [STAGING DISABLED] Would advance to stage '
                      f'{gravity_stage_idx + 1} (g={GRAVITY_STAGES[gravity_stage_idx + 1]:.2f} m/s^2) '
                      f'— set ENABLE_GRAVITY_STAGING=True to activate')
                gravity_streak = 0

        if pinch_ok and gravity_stage_idx == len(GRAVITY_STAGES) - 1 and it > 300:
            print(f'\nPinch converged at full gravity — iteration {it}!')
            break

final_q = state[:NQ_TOTAL]
final_v = state[NQ_TOTAL:]

pin.forwardKinematics(f_model, data_full, final_q)

oMi_if = data_full.oMi[INDEX_TIP_JOINT_ID]
oMi_tf = data_full.oMi[THUMB_TIP_JOINT_ID]
p_i_f  = oMi_if.translation + oMi_if.rotation @ INDEX_CONTACT_OFFSET_LOCAL
p_th_f = oMi_tf.translation + oMi_tf.rotation @ THUMB_CONTACT_OFFSET_LOCAL
tgt_if = index_target_world(final_q)
tgt_tf = thumb_target_world(final_q)
ei_f   = float(np.linalg.norm(p_i_f  - tgt_if))
eth_f  = float(np.linalg.norm(p_th_f - tgt_tf))
g_a_f  = p_i_f - p_th_f
g_d_f  = tgt_if - tgt_tf
pg_f   = float(np.linalg.norm(g_a_f - g_d_f))
vc_f   = float(np.linalg.norm(final_v[NV_HAND     : NV_HAND + 3]))
wc_f   = float(np.linalg.norm(final_v[NV_HAND + 3 : NV_HAND + 6]))
drift_f = float(np.linalg.norm(cube_position_from_q(final_q) - cube_pos0))
R_f = cube_rotation_from_q(final_q)
cf  = (np.trace(cube_rotation_from_q(q0).T @ R_f) - 1) / 2
rot_f = float(np.degrees(np.arccos(np.clip(cf, -1.0, 1.0))))

update_viser_from_q(final_q)
_update_viser_dots(final_q)

print(f'Gravity stage : {gravity_stage_idx}  g={current_gravity_z:.2f} m/s^2')
print(f'e_i  = {ei_f*1e3:.2f} mm   (tol {PINCH_E_TOL*1e3:.0f} mm)')
print(f'e_th = {eth_f*1e3:.2f} mm   (tol {PINCH_E_TOL*1e3:.0f} mm)')
print(f'pinch_err = {pg_f*1e3:.2f} mm   (tol {PINCH_G_TOL*1e3:.0f} mm)')
print(f'vc  = {vc_f:.4f} m/s   (tol {CUBE_VC_TOL:.3f} m/s)')
print(f'wc  = {wc_f:.4f} rad/s  (tol {CUBE_WC_TOL:.2f} rad/s)')
print(f'cube drift = {drift_f*1e3:.2f} mm   rot = {rot_f:.2f} deg')
print(f'Index joints : {np.round(final_q[INDEX_IDX], 4)}')
print(f'Thumb  joints : {np.round(final_q[THUMB_IDX], 4)}')
print(f'Cube pos : {np.round(final_q[NQ_HAND:], 4)}')
print(final_v)


Stabilizing for 2 s (gravity off, holding pose) ...
Stabilization done (400 steps).
  Cube pos : [-0.0576 -0.0296  0.192 ]
  Max |v|  : 0.0000 rad/s
Viser: green=index pad, blue=thumb pad, orange=index target, purple=thumb target

Starting MPPI: K=50  T=60  N_exec=1  workers=16
Gravity stage 0/3: g=0.0 m/s^2
[  0] g=0.0  S(0/20)  e_i= 12.9mm  e_th= 23.7mm  pinch= 35.4mm  vc=0.000m/s  wc=0.000rad/s  rot=0.0deg  best=1785398.9  mean=6436412.3  17.9Hz
[  1] g=0.0  S(0/20)  e_i= 12.9mm  e_th= 23.7mm  pinch= 35.4mm  vc=0.000m/s  wc=0.000rad/s  rot=0.0deg  best=724035.1  mean=3395505.8  13.6Hz
[  2] g=0.0  S(0/20)  e_i= 12.8mm  e_th= 23.4mm  pinch= 34.9mm  vc=0.000m/s  wc=0.000rad/s  rot=0.0deg  best=535136.0  mean=1819580.7  12.7Hz
[  3] g=0.0  S(0/20)  e_i= 12.6mm  e_th= 22.7mm  pinch= 33.9mm  vc=0.000m/s  wc=0.000rad/s  rot=0.0deg  best=422595.3  mean=1432665.0  11.3Hz
[  4] g=0.0  S(0/20)  e_i= 12.2mm  e_th= 21.5mm  pinch= 32.2mm  vc=0.000m/s  wc=0.000rad/s  rot=0.0deg  best=234857.3  me